# Comparing Engines and Policies in PropFlow

This notebook demonstrates how to use different BP engine variants and policies to improve convergence and solution quality.

## What You'll Learn

1. Understanding engine policies (damping, splitting, cost reduction, etc.)
2. Running different engines on the same problem
3. Using the Simulator for batch experiments
4. Comparing convergence behavior visually
5. When to use which policy

## What are Policies?

**Policies** modify the standard BP algorithm to improve performance:

- **Damping**: Smooths messages over iterations to prevent oscillations
- **Factor Splitting**: Splits factors to alter information flow
- **Cost Reduction**: Applies one-time cost discounts
- **Message Pruning**: Limits the number of active messages
- **TRW (Tree-Reweighted)**: Provides convergence guarantees

In PropFlow, policies are implemented as specialized engine classes.

In [ ]:
# imports
import numpy as np
import matplotlib.pyplot as plt
from propflow import (
    BPEngine,
    DampingEngine,
    SplitEngine,
    DiffusionEngine,
    CostReductionOnceEngine,
    DampingCROnceEngine,
    MessagePruningEngine,
    FGBuilder,
    CTFactories,
    MinSumComputator,
    Simulator,
)

print("PropFlow imported successfully!")

# set random seed for reproducibility
np.random.seed(42)

## Create a Test Problem

Let's create a challenging cycle graph that's prone to oscillations - perfect for testing different policies.

In [ ]:
# create a moderately sized cycle graph
test_graph = FGBuilder.build_cycle_graph(
    num_vars=8,
    domain_size=6,
    ct_factory=CTFactories.RANDOM_INT.fn,
    ct_params={'low': 0, 'high': 100},
)

print(f"Created cycle graph:")
print(f"  Variables: {len(test_graph.variables)}")
print(f"  Factors: {len(test_graph.factors)}")
print(f"  Domain size: {test_graph.variables[0].domain}")

## Policy 1: Standard BP (Baseline)

In [ ]:
# standard BP with no policies
engine_standard = BPEngine(
    factor_graph=test_graph,
    computator=MinSumComputator()
)
engine_standard.run(max_iter=100)

print(f"Standard BP:")
print(f"  Converged: {engine_standard.converged}")
print(f"  Final cost: {engine_standard.calculate_global_cost():.2f}")
print(f"  Iterations: {engine_standard.step}")

## Policy 2: Damping

Damping smooths messages to prevent oscillations:

```
message_new = damping_factor * message_new + (1 - damping_factor) * message_old
```

Typical damping factors: 0.5-0.9

In [ ]:
# try different damping factors
damping_factors = [0.3, 0.5, 0.7, 0.9]
damping_results = {}

for df in damping_factors:
    # need to create fresh graph for each engine
    graph = FGBuilder.build_cycle_graph(
        num_vars=8, domain_size=6,
        ct_factory=CTFactories.RANDOM_INT.fn,
        ct_params={'low': 0, 'high': 100},
    )
    
    engine = DampingEngine(
        factor_graph=graph,
        computator=MinSumComputator(),
        damping_factor=df
    )
    engine.run(max_iter=100)
    
    damping_results[df] = {
        'costs': engine.history.costs,
        'final_cost': engine.calculate_global_cost(),
        'converged': engine.converged,
        'iterations': engine.step
    }
    
    print(f"Damping factor {df}: cost={damping_results[df]['final_cost']:.2f}, "
          f"converged={damping_results[df]['converged']}, iters={damping_results[df]['iterations']}")

In [ ]:
# visualize damping effect
plt.figure(figsize=(12, 6))

# plot standard BP
plt.plot(engine_standard.history.costs, label='Standard BP', linewidth=2, color='black', linestyle='--')

# plot different damping factors
colors = ['blue', 'green', 'orange', 'red']
for (df, result), color in zip(damping_results.items(), colors):
    plt.plot(result['costs'], label=f'Damping {df}', linewidth=2, color=color, alpha=0.8)

plt.xlabel('Iteration', fontsize=12)
plt.ylabel('Global Cost', fontsize=12)
plt.title('Effect of Damping on Convergence', fontsize=14)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Observation

Higher damping factors (e.g., 0.9) create smoother convergence but may converge slower. Lower values (0.3) react faster but might oscillate more.

## Policy 3: Factor Splitting

Factor splitting divides each factor into two virtual factors with split cost tables. This can help information flow differently through the graph.

In [ ]:
# create fresh graph for splitting
split_graph = FGBuilder.build_cycle_graph(
    num_vars=8, domain_size=6,
    ct_factory=CTFactories.RANDOM_INT.fn,
    ct_params={'low': 0, 'high': 100},
)

# split engine
engine_split = SplitEngine(
    factor_graph=split_graph,
    computator=MinSumComputator(),
    split_factor=0.5  # 50-50 split
)
engine_split.run(max_iter=100)

print(f"Split Engine:")
print(f"  Converged: {engine_split.converged}")
print(f"  Final cost: {engine_split.calculate_global_cost():.2f}")
print(f"  Iterations: {engine_split.step}")

## Policy 4: Cost Reduction

Applies a one-time multiplicative reduction to cost tables at initialization. Can help escape local minima.

In [ ]:
# create fresh graph
cr_graph = FGBuilder.build_cycle_graph(
    num_vars=8, domain_size=6,
    ct_factory=CTFactories.RANDOM_INT.fn,
    ct_params={'low': 0, 'high': 100},
)

# cost reduction engine
engine_cr = CostReductionOnceEngine(
    factor_graph=cr_graph,
    computator=MinSumComputator(),
    cost_reduction_factor=0.8  # reduce costs by 20%
)
engine_cr.run(max_iter=100)

print(f"Cost Reduction Engine:")
print(f"  Converged: {engine_cr.converged}")
print(f"  Final cost: {engine_cr.calculate_global_cost():.2f}")
print(f"  Iterations: {engine_cr.step}")

## Policy 5: Combined Policies

PropFlow allows combining multiple policies. Let's use damping + cost reduction together.

In [ ]:
# create fresh graph
combined_graph = FGBuilder.build_cycle_graph(
    num_vars=8, domain_size=6,
    ct_factory=CTFactories.RANDOM_INT.fn,
    ct_params={'low': 0, 'high': 100},
)

# damping + cost reduction engine
engine_combined = DampingCROnceEngine(
    factor_graph=combined_graph,
    computator=MinSumComputator(),
    damping_factor=0.7,
    cost_reduction_factor=0.85
)
engine_combined.run(max_iter=100)

print(f"Combined (Damping + Cost Reduction) Engine:")
print(f"  Converged: {engine_combined.converged}")
print(f"  Final cost: {engine_combined.calculate_global_cost():.2f}")
print(f"  Iterations: {engine_combined.step}")

## Using the Simulator for Batch Experiments

The **Simulator** allows you to:
1. Compare multiple engine configurations
2. Run on multiple problem instances
3. Execute in parallel
4. Automatically plot averaged results

In [ ]:
# create multiple problem instances
num_graphs = 10
graphs = [
    FGBuilder.build_cycle_graph(
        num_vars=10,
        domain_size=5,
        ct_factory=CTFactories.RANDOM_INT.fn,
        ct_params={'low': 50, 'high': 150},
    )
    for _ in range(num_graphs)
]

print(f"Created {num_graphs} random cycle graphs for testing")

In [ ]:
# define engine configurations to compare
engine_configs = {
    "Standard BP": {
        "class": BPEngine,
        "computator": MinSumComputator()
    },
    "Damping (0.7)": {
        "class": DampingEngine,
        "computator": MinSumComputator(),
        "damping_factor": 0.7
    },
    "Damping (0.9)": {
        "class": DampingEngine,
        "computator": MinSumComputator(),
        "damping_factor": 0.9
    },
    "Split (0.5)": {
        "class": SplitEngine,
        "computator": MinSumComputator(),
        "split_factor": 0.5
    },
    "Damping + CR": {
        "class": DampingCROnceEngine,
        "computator": MinSumComputator(),
        "damping_factor": 0.7,
        "cost_reduction_factor": 0.85
    },
}

print(f"Configured {len(engine_configs)} engine variants")

In [ ]:
# create and run simulator
simulator = Simulator(
    engine_configs=engine_configs,
    log_level="MINIMAL"  # reduce console output
)

# run simulations
print("Running simulations...")
results = simulator.run_simulations(
    graphs=graphs,
    max_iter=200
)

print("\nSimulations complete!")

In [ ]:
# plot comparison results
simulator.plot_results(
    verbose=True,  # show detailed statistics
    show_std=True  # show standard deviation bands
)

## Analyzing Results

Let's extract and compare specific metrics from the simulation.

In [ ]:
# extract final costs for each engine
final_costs = {}
convergence_rates = {}

for engine_name in engine_configs.keys():
    # get results for this engine across all graphs
    engine_results = [r for r in results if r['engine_name'] == engine_name]
    
    # extract final costs
    costs = [r['final_cost'] for r in engine_results]
    final_costs[engine_name] = {
        'mean': np.mean(costs),
        'std': np.std(costs),
        'min': np.min(costs),
        'max': np.max(costs)
    }
    
    # convergence rate
    converged_count = sum(1 for r in engine_results if r.get('converged', False))
    convergence_rates[engine_name] = converged_count / len(engine_results) * 100

# print summary
print("\n" + "="*60)
print("FINAL COST SUMMARY")
print("="*60)
for engine_name, stats in final_costs.items():
    print(f"\n{engine_name}:")
    print(f"  Mean: {stats['mean']:.2f} ± {stats['std']:.2f}")
    print(f"  Range: [{stats['min']:.2f}, {stats['max']:.2f}]")
    print(f"  Convergence rate: {convergence_rates[engine_name]:.1f}%")

In [ ]:
# create comparison bar chart
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# plot mean final costs
engine_names = list(final_costs.keys())
means = [final_costs[name]['mean'] for name in engine_names]
stds = [final_costs[name]['std'] for name in engine_names]

ax1.bar(range(len(engine_names)), means, yerr=stds, capsize=5, alpha=0.7, color='steelblue')
ax1.set_xticks(range(len(engine_names)))
ax1.set_xticklabels(engine_names, rotation=45, ha='right')
ax1.set_ylabel('Final Cost', fontsize=11)
ax1.set_title('Mean Final Cost by Engine', fontsize=13)
ax1.grid(True, alpha=0.3, axis='y')

# plot convergence rates
conv_rates = [convergence_rates[name] for name in engine_names]
ax2.bar(range(len(engine_names)), conv_rates, alpha=0.7, color='coral')
ax2.set_xticks(range(len(engine_names)))
ax2.set_xticklabels(engine_names, rotation=45, ha='right')
ax2.set_ylabel('Convergence Rate (%)', fontsize=11)
ax2.set_title('Convergence Rate by Engine', fontsize=13)
ax2.set_ylim([0, 105])
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## When to Use Which Policy?

Based on problem characteristics:

| **Problem Type** | **Recommended Policy** | **Why** |
|------------------|------------------------|--------|
| **Oscillating messages** | Damping (0.7-0.9) | Smooths message updates |
| **Slow convergence** | Damping (0.3-0.5) or Diffusion | Faster information propagation |
| **Cycles/loops** | Factor Splitting | Breaks symmetry in loops |
| **Stuck in local minima** | Cost Reduction | Temporarily flattens landscape |
| **Large dense graphs** | Message Pruning | Reduces computational overhead |
| **Need guarantees** | TRW Engine | Convergence guarantees |
| **Unknown/complex** | Try Damping + CR | Often works well in practice |

## Advanced: Testing on Random Graphs

Let's test policies on random graphs (not just cycles) to see how they perform on different topologies.

In [ ]:
# create random graphs with varying density
random_graphs = [
    FGBuilder.build_random_graph(
        num_vars=15,
        domain_size=4,
        ct_factory=CTFactories.RANDOM_INT.fn,
        ct_params={'low': 10, 'high': 80},
        density=0.35  # 35% connectivity
    )
    for _ in range(8)
]

print(f"Created {len(random_graphs)} random graphs")
print(f"Average factors per graph: {np.mean([len(g.factors) for g in random_graphs]):.1f}")

In [ ]:
# define a subset of engines to test
random_engine_configs = {
    "Standard BP": {
        "class": BPEngine,
        "computator": MinSumComputator()
    },
    "Damping (0.8)": {
        "class": DampingEngine,
        "computator": MinSumComputator(),
        "damping_factor": 0.8
    },
    "Diffusion": {
        "class": DiffusionEngine,
        "computator": MinSumComputator(),
        "diffusion_rate": 0.3
    },
}

# run simulator
sim_random = Simulator(random_engine_configs, log_level="MINIMAL")
results_random = sim_random.run_simulations(random_graphs, max_iter=150)

# plot
sim_random.plot_results(verbose=True)

## Summary

In this tutorial, you learned:

1. ✅ Different **policies** and when to use them
2. ✅ How to configure and run **specialized engines**
3. ✅ Using the **Simulator** for batch experiments
4. ✅ Comparing engine performance **quantitatively**
5. ✅ Visualizing convergence behavior

## Next Steps

- **Tutorial 03**: Apply PropFlow to real-world problems (graph coloring, scheduling)
- Experiment with **snapshot analysis** for deeper debugging
- Try **search-based algorithms** (DSA, MGM) for comparison
- Check the [PropFlow documentation](https://ormullerhahitti.github.io/Belief-Propagation-Simulator/) for advanced features